# 01 - Dataset Visualization

Part 1 (prepare & visualize) of the course project pipeline. Downloads/caches
the KITTI subsets (object detection, semantics, optical flow) and visualizes
sample images with their native ground truth.

See `CLAUDE.md` for the locked dataset/task/distortion decisions.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.getcwd())

## Load KITTI subsets

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from ipproj import config
from ipproj.datasets import kitti, kitti_flow
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.viz.plotting import plot_detection_boxes, plot_segmentation_mask, plot_image_grid, save_figure

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

print("object_detection:", {k: len(v) for k, v in detection_splits.items()})
print("semantic_segmentation:", {k: len(v) for k, v in segmentation_splits.items()})
print("optical_flow:", {k: len(v) for k, v in flow_splits.items()})

## Sample grid: object detection ground truth

Every figure in this notebook is saved as a PNG under `figures/` (via
`save_figure`) in addition to displaying inline - `figures/` is tracked in
git, so these are ready to embed straight into the README
(`![caption](figures/01_dataset_visualization/detection_gt_grid.png)`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, sample in zip(axes.flat, detection_splits["train"][:4]):
    image = read_image(sample.image_path)
    plot_detection_boxes(image, sample.boxes, sample.classes, ax=ax)
fig.suptitle("KITTI object detection - clean images with GT boxes")
fig.tight_layout()
save_figure(fig, "01_dataset_visualization/detection_gt_grid.png")

## Sample grid: semantic segmentation ground truth

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, sample in zip(axes.flat, segmentation_splits["train"][:4]):
    image = read_image(sample.image_path)
    mask = cv2.imread(str(sample.mask_path), cv2.IMREAD_UNCHANGED)
    plot_segmentation_mask(image, mask, ax=ax)
fig.suptitle("KITTI Semantics - clean images with GT masks")
fig.tight_layout()
save_figure(fig, "01_dataset_visualization/segmentation_gt_grid.png")

## Sample pair: optical flow ground truth

In [ ]:
sample = flow_splits["train"][0]
frame1 = read_image(sample.frame1_path)
frame2 = read_image(sample.frame2_path)
flow, valid = read_kitti_flow_png(sample.flow_gt_path)

hsv = np.zeros_like(frame1)
hsv[..., 1] = 255
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv[..., 0] = ang * 180 / np.pi / 2
hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
flow_rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

fig = plot_image_grid([frame1, frame2, flow_rgb], titles=["Frame t", "Frame t+1", "GT flow (color wheel)"], ncols=3)
save_figure(fig, "01_dataset_visualization/optical_flow_gt.png")

## Dataset statistics

In [ ]:
from collections import Counter

class_counts = Counter(cls for s in detection_splits["train"] for cls in s.classes)
detection_class_stats = pd.DataFrame(sorted(class_counts.items()), columns=["class", "count"])
detection_class_stats

In [ ]:
split_summary = pd.DataFrame({
    "task": ["object_detection", "semantic_segmentation", "optical_flow"],
    "train": [len(detection_splits["train"]), len(segmentation_splits["train"]), len(flow_splits["train"])],
    "val": [len(detection_splits["val"]), len(segmentation_splits["val"]), len(flow_splits["val"])],
    "test": [len(detection_splits["test"]), len(segmentation_splits["test"]), len(flow_splits["test"])],
})
split_summary